# Decorators

A decorator is a function that manipulates another function or class. It is a very useful feature in the Python language.

In these practical exercises, we will learn how to use decorators.

## Function Decorators: Functions That Manipulate Functions

Write a function `squared` that takes a function as input and returns that same function with its result squared:

- the original function always takes two arguments
- the original function always produces a number
- the squared function takes **a function** as input and produces one as output

In [ ]:
def add(a, b):
  return a + b


def sub(a, b):
  return a - b


def mul(a, b):
  return a * b


def squared(function):
  return function  # Your code here


print(squared(add)(2, 2))  # Should be 16
print(squared(sub)(10, 5))  # Should be 25
print(squared(mul)(3, 4))  # Should be 144

### Additional hints

`squared` defines a new function of two arguments, which calls `function` and returns the square of its result, then returns this new function, without calling it. `squared(add)` is therefore a function, and `squared(add)(2, 2)` calls it.

```python
def squared(function):
  def new_function(a, b):
    return ...

  return ...
```

### Solution

In [ ]:
def squared(function):
  def new_function(a, b):
    return function(a, b) ** 2

  return new_function


print(squared(add)(2, 2))
print(squared(sub)(10, 5))
print(squared(mul)(3, 4))

## Order of Stacked Decorators

The two decorators below print a message before and after calling the function they decorate, and wrap its result in an HTML tag.

Without running any code, write down, in order, the lines that `print(greet("Ada"))` will print. Then check your answer by running both cells.

In [ ]:
import functools


def bold(function):
  @functools.wraps(function)
  def new_function(*args, **kwargs):
    print("bold: before")
    result = function(*args, **kwargs)
    print("bold: after")
    return f"<b>{result}</b>"

  return new_function


def italic(function):
  @functools.wraps(function)
  def new_function(*args, **kwargs):
    print("italic: before")
    result = function(*args, **kwargs)
    print("italic: after")
    return f"<i>{result}</i>"

  return new_function


@bold
@italic
def greet(name):
  print("greet")
  return f"Hello {name}"

In [ ]:
print(greet("Ada"))

*Your answer*

### Additional hints

`@bold` above `@italic` is equivalent to `greet = bold(italic(greet))`. Each decorator prints "before", calls the function it wraps, then prints "after": `print` only displays the result once all these calls are over.

```text
bold: before
...
<b>...</b>
```

### Solution

```text
bold: before
italic: before
greet
italic: after
bold: after
<b><i>Hello Ada</i></b>
```

`@bold @italic def greet` is equivalent to `greet = bold(italic(greet))`: `bold` wraps `italic`, which wraps `greet`. The call goes through the layers from the outside in, and the result comes back out in reverse order.

## Measuring a Function's Execution Time

Create a `timeit` decorator that displays a function's execution time.

**Useful documentation**

- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)

In [ ]:
# Your code here

### Additional hints

Read `time.perf_counter()` before and after calling the function: the duration in seconds is their difference. The inner function accepts `*args` and `**kwargs`, and returns the decorated function's result.

```python
import time


def timeit(function):
  def new_function(*args, **kwargs):
    start = ...
    result = ...
    print(...)
    return result

  return new_function
```

### Solution

In [ ]:
import time


def timeit(function):
  def new_function(*args, **kwargs):
    start = time.perf_counter()
    result = function(*args, **kwargs)
    print(
      f"Time elapsed during '{function.__name__}' call: {time.perf_counter() - start}s"
    )
    return result

  return new_function


@timeit
def add(a, b):
  return a + b


add(1, 2)

## Plugin Registry

A registry lets a library use client code it doesn't know, in three steps:

1. The library provides a decorator, which records a class or a function.
2. The client code applies this decorator.
3. The library uses the recorded items when it needs them.

The next cells create a `plugins/` folder with two client code modules, `format_csv.py` and `format_json.py`, each recording an export class. Write the library side:

1. In the `registry.py` module, complete the `exporter(name)` class decorator, which records the class in `exporters` under the key `name`, and raises a `ValueError` if this name is already taken.
2. Write a `load_plugins(folder)` function, which imports all the modules of `folder`.
3. Print `registry.exporters` before and after calling `load_plugins("plugins")`. What do you notice?
4. Export `[[1, 2], [3, 4]]` as CSV through the registry, then check that recording another class under the name `"csv"` does raise a `ValueError`.

**Useful documentation**

- [`pkgutil.iter_modules`](https://docs.python.org/3/library/pkgutil.html#pkgutil.iter_modules)
- [`importlib.import_module`](https://docs.python.org/3/library/importlib.html#importlib.import_module)

In [ ]:
from pathlib import Path

Path("plugins").mkdir(exist_ok=True)

In [ ]:
%%writefile plugins/format_csv.py
from registry import exporter


@exporter("csv")
class CsvExporter:
  def export(self, rows):
    return "\n".join(",".join(map(str, row)) for row in rows)

In [ ]:
%%writefile plugins/format_json.py
import json

from registry import exporter


@exporter("json")
class JsonExporter:
  def export(self, rows):
    return json.dumps(rows)

In [ ]:
%%writefile registry.py
exporters = {}


def exporter(name):
  ...  # Your code here

In [ ]:
# Your code here

*Your answer*

### Additional hints

`exporter(name)` returns a decorator, which receives the class, stores it in `exporters` and returns it. `pkgutil.iter_modules([folder])` goes through a folder's modules, each with its `name` attribute, and `importlib.import_module("plugins.format_csv")` imports a module from its name. Before the import, the registry is `{}`; after it, it has the keys `'csv'` and `'json'`. The export gives two lines, `1,2` and `3,4`.

Python only imports a module once: after fixing `registry.py`, restart the session before trying again.

The `registry.py` module:

```python
exporters = {}


def exporter(name):
  def decorator(cls):
    if ...:
      raise ValueError(...)
    exporters[name] = ...
    return ...

  return ...
```

The rest, in the next cell:

```python
import importlib
import pkgutil

import registry


def load_plugins(folder):
  for module in pkgutil.iter_modules([folder]):
    importlib.import_module(...)


print(registry.exporters)
load_plugins("plugins")
print(registry.exporters)
print(registry.exporters["csv"]().export(...))

try:

  @registry.exporter("csv")
  class OtherExporter:
    pass

except ValueError as error:
  print(error)
```

### Solution

In [ ]:
%%writefile registry.py
exporters = {}


def exporter(name):
  def decorator(cls):
    if name in exporters:
      raise ValueError(f"Name already taken: {name}")
    exporters[name] = cls
    return cls

  return decorator

In [ ]:
import importlib
import pkgutil

import registry


def load_plugins(folder):
  for module in pkgutil.iter_modules([folder]):
    importlib.import_module(f"{folder}.{module.name}")


print(registry.exporters)
load_plugins("plugins")
print(registry.exporters)
print(registry.exporters["csv"]().export([[1, 2], [3, 4]]))

try:

  @registry.exporter("csv")
  class OtherExporter:
    pass

except ValueError as error:
  print(error)

The registry stays empty as long as the modules of `plugins/` aren't imported: a decorator runs when the class is defined, so when its module is imported. A library relying on a registry must therefore import the client code before using it: that's the role of `load_plugins`.

This pattern is called *Registry*.

## Subscribing to Events

A shop wants to trigger several actions when an order is confirmed (sending a confirmation, updating the stock…), without the code confirming the order knowing these actions. This is the *observer* pattern, written here with a decorator.

Write:

- a `subscribers` dictionary, which maps each event to the list of its subscribed functions;
- a decorator with an argument, `subscribe(event)`, which adds the decorated function to the subscribers of `event`;
- a `publish(event, *args)` function, which calls with `args`, in subscription order, all the subscribers of `event`.

Subscribe two functions to the `"order_confirmed"` event, `send_confirmation` and `update_stock`, then publish it for the order `{"customer": "Ada", "quantity": 2}`: both messages must be printed. Publishing an event without subscribers, such as `"order_cancelled"`, must do nothing.

**Useful documentation**

- [`dict.setdefault`](https://docs.python.org/3/library/stdtypes.html#dict.setdefault)

In [ ]:
# Your code here

### Additional hints

Like `exporter`, `subscribe` returns a decorator, which returns the function as is: it stays directly callable. `subscribers.setdefault(event, [])` returns the list of subscribers, created if needed, and `subscribers.get(event, [])` avoids a `KeyError` for an event without subscribers. You should get `Confirmation for Ada`, then `Stock reduced by 2`.

```python
subscribers = {}


def subscribe(event):
  def decorator(function):
    subscribers.setdefault(event, []).append(...)
    return ...

  return ...


def publish(event, *args):
  for function in ...:
    ...


@subscribe("order_confirmed")
def send_confirmation(order):
  print("Confirmation for", order["customer"])
```

### Solution

In [ ]:
subscribers = {}


def subscribe(event):
  def decorator(function):
    subscribers.setdefault(event, []).append(function)
    return function

  return decorator


def publish(event, *args):
  for function in subscribers.get(event, []):
    function(*args)


@subscribe("order_confirmed")
def send_confirmation(order):
  print("Confirmation for", order["customer"])


@subscribe("order_confirmed")
def update_stock(order):
  print("Stock reduced by", order["quantity"])


publish("order_confirmed", {"customer": "Ada", "quantity": 2})
publish("order_cancelled", {"customer": "Ada"})

## Using `functools.wraps`

Decorate a documented function with your `timeit`, then print its name (`__name__` attribute) and its documentation (`__doc__` attribute). What do you notice?

Then use `functools.wraps` to solve this issue.

In [ ]:
# Your code here

*Your answer*

### Additional hints

The decorated function is `new_function`: its name and documentation are what you are reading. `@functools.wraps(function)` goes right above `def new_function`.

```python
import functools
import time


def timeit(function):
  @functools.wraps(...)
  def new_function(*args, **kwargs): ...

  return new_function


@timeit
def add(a, b):
  """..."""
  return a + b


print(add.__name__, add.__doc__)
```

### Solution

In [ ]:
import functools
import time


def timeit(function):
  @functools.wraps(function)
  def new_function(*args, **kwargs):
    start = time.perf_counter()
    result = function(*args, **kwargs)
    print(
      f"Time elapsed during '{function.__name__}' call: {time.perf_counter() - start}s"
    )
    return result

  return new_function


@timeit
def add(a, b):
  """Add two integers."""
  return a + b


print(add.__name__, add.__doc__)

Without `functools.wraps`, `__name__` is the inner function's name (`new_function`) and `__doc__` is `None`: the decorated function has lost its identity. With `functools.wraps`, it gets its name and documentation back.

## `timeit` Decorator with an Argument

Create a `timeit` decorator that takes a `unit` argument (`"s"` or `"ms"`), the unit in which to display the execution time, and that keeps `functools.wraps`.

In [ ]:
# Your code here

### Additional hints

There are three levels of functions: `timeit(unit)` returns `decorator`, which receives the function and returns `new_function`. `@timeit(unit="ms")` first calls `timeit`, then applies what it returns to the decorated function. A duration in seconds multiplied by 1000 gives milliseconds.

```python
import functools
import time


def timeit(unit="s"):
  def decorator(function):
    @functools.wraps(function)
    def new_function(*args, **kwargs):
      start = time.perf_counter()
      result = ...
      duration = ...
      if unit == "ms":
        ...
      print(...)
      return result

    return ...

  return ...
```

### Solution

In [ ]:
import functools
import time


def timeit(unit="s"):
  if unit not in ("s", "ms"):
    raise ValueError("Can only use s or ms as unit argument")

  def decorator(function):
    @functools.wraps(function)
    def new_function(*args, **kwargs):
      start = time.perf_counter()
      result = function(*args, **kwargs)
      duration = time.perf_counter() - start
      if unit == "ms":
        duration *= 1000
      print(f"Time elapsed during '{function.__name__}' call: {duration}{unit}")
      return result

    return new_function

  return decorator


@timeit(unit="ms")
def add(a, b):
  """Add two integers."""
  return a + b


print(add.__name__, add.__doc__)

## Caching Results

Write a `memoize` decorator that stores the result of each call in a dictionary, keyed by its arguments, and returns it directly when the function is called again with the same arguments.

Then use your `timeit` decorator to compare how long `fibonacci(32)` takes without a cache, with `memoize`, and with `functools.cache`. To do so:

- write three distinct recursive functions, each calling itself, the first one without a decorator, the other two decorated with `@`;
- only measure the outer call, with `timeit(unit="ms")(function)(32)`, and don't apply `@timeit` to a recursive function.

**Useful documentation**

- [`functools.cache`](https://docs.python.org/3/library/functools.html#functools.cache)

In [ ]:
def fibonacci(n: int) -> int:
  return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)

In [ ]:
# Your code here

### Additional hints

The dictionary is created in `memoize`: the inner function keeps it through a closure, from one call to the next. `args` is a tuple, hence hashable: it directly serves as a key. `fibonacci(32)` is 2178309.

```python
import functools


def memoize(function):
  cache = {}

  @functools.wraps(function)
  def new_function(*args):
    if ...:
      cache[args] = ...
    return ...

  return new_function


@memoize
def fibonacci(n: int) -> int:
  return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)
```

### Solution

In [ ]:
import functools


def memoize(function):
  cache = {}

  @functools.wraps(function)
  def new_function(*args):
    if args not in cache:
      cache[args] = function(*args)
    return cache[args]

  return new_function


def fibonacci_plain(n: int) -> int:
  return n if n < 2 else fibonacci_plain(n - 1) + fibonacci_plain(n - 2)


@memoize
def fibonacci(n: int) -> int:
  return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)


@functools.cache
def fibonacci_cache(n: int) -> int:
  return n if n < 2 else fibonacci_cache(n - 1) + fibonacci_cache(n - 2)


for function in (fibonacci_plain, fibonacci, fibonacci_cache):
  timeit(unit="ms")(function)(32)

Without a cache, `fibonacci(32)` recomputes the same values millions of times; with one, each value is computed once. `functools.cache` does the same as `memoize`, also handles keyword arguments, and provides `cache_info()` and `cache_clear()`.

Two traps come from `@` reassigning the function's name:

- `@timeit` on a recursive function also decorates its recursive calls, which go through this name: `fibonacci(32)` would print one line for each of its 7,049,155 calls. Hence `timeit(unit="ms")(function)(32)`, which only measures the outer call.
- `memoize(fibonacci)(32)`, without reassigning the name, gains almost nothing: inside, the recursion still calls the original function, without a cache. That's why the function is decorated with `@`.